# Education Progress Forecast

## 1) Problem Framing
**Business question:** Which residents are likely to show stalled or low future education progress?

- **Predictive:** Regress or classify next-period `progress_percent` / stall risk.
- **Explanatory:** Which health, counseling, and demographic factors associate with progress?

**Metrics:** MAE, R² for regression; ROC-AUC if binary stall.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

edu = pd.read_csv(DATA_DIR / "education_records.csv", parse_dates=["record_date"])
edu = edu.sort_values(["resident_id", "record_date"])
edu["next_progress"] = edu.groupby("resident_id")["progress_percent"].shift(-1)
edu["stall"] = (edu["next_progress"] <= edu["progress_percent"]).astype(int)
row = edu.dropna(subset=["next_progress"])
residents = pd.read_csv(DATA_DIR / "residents.csv")
health = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv", parse_dates=["record_date"])
hlast = health.sort_values(["resident_id", "record_date"]).groupby("resident_id").tail(1)
hlast = hlast.rename(columns=lambda c: "h_" + c if c != "resident_id" else c)
row = row.merge(hlast, on="resident_id", how="left")
row = row.merge(
    residents[["resident_id", "present_age", "length_of_stay", "initial_risk_level", "current_risk_level", "case_category", "safehouse_id"]],
    on="resident_id",
    how="left",
)
feat = [
    "attendance_rate",
    "progress_percent",
    "education_level",
    "enrollment_status",
    "present_age",
    "length_of_stay",
    "initial_risk_level",
    "current_risk_level",
    "case_category",
    "safehouse_id",
    "h_general_health_score",
    "h_sleep_quality_score",
    "h_nutrition_score",
]
feat = [c for c in feat if c in row.columns]
X = row[feat].copy()
y = row["stall"].astype(int)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
prep = ColumnTransformer(
    [
        ("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("im", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y if y.nunique() > 1 else None)
exp_m = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
pred_m = Pipeline([("prep", prep), ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1))])
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
proba = pred_m.predict_proba(X_te)[:, 1]
print("ROC-AUC:", roc_auc_score(y_te, proba) if y_te.nunique() > 1 else "n/a")
print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))
fn = exp_m.named_steps["prep"].get_feature_names_out()
display(pd.DataFrame({"feature": fn, "coef": exp_m.named_steps["clf"].coef_[0]}).assign(abs=lambda d: d["coef"].abs()).sort_values("abs", ascending=False).head(12))


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_generic_classifier_dashboard, save_dashboard_json
_fn = exp_m.named_steps["prep"].get_feature_names_out()
_coef = pd.DataFrame({"feature": _fn, "coef": exp_m.named_steps["clf"].coef_[0]})
_coef["abs_coef"] = _coef["coef"].abs()
_imp = pd.DataFrame({"feature": _fn, "importance": pred_m.named_steps["clf"].feature_importances_})
_dash = export_generic_classifier_dashboard(
    pipeline_id="education-progress-forecast",
    display_name="Education progress forecast",
    problem_summary="Flags residents at risk of stalled education progress (next-period stall proxy).",
    pred_m=pred_m,
    exp_m=exp_m,
    X_te=X_te,
    y_te=y_te,
    proba=proba,
    coef_df=_coef,
    imp_df=_imp,
    positive_class_description="stalled or flat progress versus prior record",
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


## 2–6) Sections
EDA: plot distributions of `progress_percent` and `attendance_rate`. **Deployment:** education dashboard with stall-risk score.
